In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import copy
import math

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

torch.manual_seed(42)
np.random.seed(42)
print("GRPO notebook 环境就绪")

GRPO notebook 环境就绪


# Cell 1：动机（一）—— PPO 用于 LLM 的挑战

PPO 在机器人控制、游戏等连续控制任务上取得了巨大成功，但将其直接应用于大语言模型（LLM）的 RLHF 训练时，会遭遇以下几个棘手问题：

---

## 1. 内存压力：Critic 网络规模问题

PPO 依赖一个 **Critic 网络（价值函数 $V_\phi$）** 来估计每个状态的期望回报。为了让 Critic 真正理解语言语义，它通常需要与 Actor（策略网络）同等规模的 Transformer。

- 7B 参数的 Actor + 7B 参数的 Critic = **14B 参数同时常驻 GPU**
- 加上 ref_model（冻结）和 policy_old，峰值可达 **28B+ 参数**
- 对于 70B 模型，这在实践中几乎不可接受

## 2. LLM Critic 的噪声问题

PPO 的 Critic 需要估计 **token-level 的价值**（每生成一个 token 之后，期望累计回报是多少）。

- 但 LLM 的生成是一个超长序列决策过程（一句话可能有几百个 token）
- 中间 token 的价值极难估计：「这个句子的第 37 个 token 对最终答案质量贡献了多少？」
- 结果是 Critic 的估计噪声极大，导致优势估计（Advantage Estimation）不稳定

## 3. 采样代价：在线 rollout 的计算负担

PPO 是 **在线算法**：每次梯度更新之前，需要用当前策略生成新的完整响应（rollout）。

- 对 LLM 来说，生成一条完整响应（比如 512 token 的数学推理步骤）本身就很耗时
- GRPO 每个 prompt 还需要采样 G 个响应，采样开销随 G 线性增长
- 但这是 **所有在线 RL 方法共有的代价**，GRPO 并不比 PPO 更糟（反而因去掉 Critic 更好）

---

这些问题促使研究者思考：**能否去掉 Critic 网络？**

> 关键问题：PPO 需要 Critic 的根本原因是什么？
> 
> 答案：需要一个 **基准线（baseline）** 来减小梯度方差。Critic 估计的 $V(s)$ 就是这个基准线。
> 
> 如果我们能找到一个**不需要神经网络的基准线**，就可以去掉 Critic。

# Cell 2：动机（二）—— DPO 的局限

DPO（Direct Preference Optimization）提供了另一种去掉 Critic 的思路：直接在偏好数据上做监督学习，完全绕开强化学习。

---

## DPO 的优雅之处

DPO 将 RLHF 的目标函数重新参数化，推导出一个纯监督学习的目标：

$$L_{DPO}(\theta) = -\mathbb{E}_{(x,y_w,y_l) \sim \mathcal{D}}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right]$$

- 不需要 Critic，不需要在线采样
- 一次前向传播，计算两个 log prob 之差，梯度直接反传
- 实现极其简洁（03_DPO.ipynb 中已详细推导）

---

## DPO 的关键局限：需要成对偏好数据

DPO 的公式中有 $(y_w, y_l)$——**同一 prompt 的两个响应，且已知哪个更好**。

但很多重要场景根本没有这样的数据：

| 场景 | 可用信号 | 能用 DPO 吗？ |
|------|----------|---------------|
| 数学题求解 | 答案对/错（0/1） | 需要构造偏好对，不自然 |
| 代码生成 | 测试通过率（0~100%） | 同上 |
| 在线实时反馈 | 用户点赞/点踩 | 需要实时配对，延迟高 |
| 游戏/控制 | 累计奖励（scalar） | 完全无法使用 |

更深层的问题：

1. **DPO 是离线的（offline）**：使用预先收集的静态数据集，训练过程中不生成新数据。这意味着如果 policy 漂离了数据分布，训练信号就会失效。

2. **离线数据覆盖问题**：数据集中没有的响应类型，DPO 无法学习如何评价。

---

## 引出核心问题

我们想要一种方法：
- ✅ 不需要 Critic（解决内存和噪声问题）
- ✅ 支持 scalar 奖励信号（不需要成对偏好数据）
- ✅ 在线训练（每步生成新数据，避免分布偏移）
- ✅ 训练稳定（类似 PPO-Clip 的保守更新）

> **这就是 GRPO（Group Relative Policy Optimization）的出发点。**

# Cell 3：GRPO 的核心洞察

## 洞察：用「组内对比」替代 Critic

PPO 需要 Critic 来回答：「这个状态/响应有多好？」

GRPO 换一个问题：「**相对于同组其他响应，这个响应更好还是更差？**」

---

## 具体机制：组相对奖励归一化

对同一个 prompt $q$，从旧策略 $\pi_{\theta_{old}}$ 采样 $G$ 个响应 $\{o_1, o_2, ..., o_G\}$，分别计算奖励 $\{R_1, R_2, ..., R_G\}$。

然后对奖励做**组内 Z-score 归一化**，得到优势估计：

$$\hat{A}_i = \frac{R_i - \text{mean}(\{R_j\}_{j=1}^G)}{\text{std}(\{R_j\}_{j=1}^G) + \epsilon}$$

其中 $\epsilon$ 是一个小常数（如 $10^{-8}$），防止除以零。

---

## 直觉解释

假设对某道数学题采样 G=4 个响应，奖励为 $[0, 0, 1, 1]$（0=答案错，1=答案对）：

- $\text{mean} = 0.5$，$\text{std} \approx 0.577$
- 两个错误答案的优势：$(0 - 0.5) / 0.577 \approx -0.87$（**应该被抑制**）
- 两个正确答案的优势：$(1 - 0.5) / 0.577 \approx +0.87$（**应该被增强**）

如果所有答案都对（奖励 $[1,1,1,1]$），则 std=0，所有优势≈0，不更新策略。这很合理：当 G 个响应都同样好时，不知道往哪个方向优化。

---

## 与 REINFORCE 的关系

GRPO 本质上是 **REINFORCE with baseline** 的变体：

- 经典 REINFORCE：$\nabla_\theta J \propto \sum_t (R - b) \nabla_\theta \log \pi_\theta(a_t|s_t)$，其中 $b$ 通常由 Critic 提供
- GRPO：用 **组内均值** $\text{mean}(R_1...R_G)$ 作为 baseline $b$，完全不需要神经网络

这个 baseline 虽然比 Critic 粗糙（Critic 是针对每个 state 定制的，组内均值是固定的），但在实践中已经足够有效，且实现极其简单。

---

## 为什么这有效？

1. **方差减小**：相对基准比绝对奖励方差更小（因为相关噪声被消除）
2. **自适应难度**：对难题（所有响应都答错）和易题（所有响应都答对）自动调整更新幅度
3. **无需标定奖励量级**：归一化后优势量级固定在 $[-3, +3]$ 左右，不受奖励函数绝对值影响

# Cell 4：数学推导 —— GRPO 目标函数

## 完整目标函数

$$L_{GRPO}(\theta) = \mathbb{E}_{q \sim p(q)}\left[\mathbb{E}_{o_i \sim \pi_{\theta_{old}}(\cdot|q),\, i=1..G}\left[\frac{1}{G}\sum_{i=1}^G \min\left(r_i \hat{A}_i,\ \text{clip}(r_i, 1-\varepsilon, 1+\varepsilon)\hat{A}_i\right) - \beta \cdot KL(\pi_\theta \| \pi_{ref})\right]\right]$$

---

## 各项拆解

### 1. 外层期望：对 prompt 采样

$$\mathbb{E}_{q \sim p(q)}[\cdots]$$

对训练集中的 prompt 分布取期望，实现上即为 mini-batch 求均值。

---

### 2. 内层期望：从旧策略采样 G 个响应

$$\mathbb{E}_{o_i \sim \pi_{\theta_{old}}(\cdot|q),\, i=1..G}[\cdots]$$

注意：采样来自 **$\pi_{\theta_{old}}$（旧策略）**，而不是当前策略 $\pi_\theta$。这是离策略（off-policy）采样，通过重要性采样比率 $r_i$ 进行修正。

---

### 3. 重要性采样比率 $r_i$

$$r_i = \frac{\pi_\theta(o_i|q)}{\pi_{\theta_{old}}(o_i|q)}$$

由于 $o_i$ 是一个 token 序列，$\pi_\theta(o_i|q) = \prod_{t} \pi_\theta(o_{i,t}|q, o_{i,<t})$。

实现上取 **per-token log prob 的均值**（而非 sum）：

$$\log r_i = \frac{1}{|o_i|}\sum_t \log \frac{\pi_\theta(o_{i,t}|\cdot)}{\pi_{\theta_{old}}(o_{i,t}|\cdot)}$$

用均值而非求和，是为了避免长序列的 $r_i$ 指数级爆炸或消失。

---

### 4. 组内归一化优势 $\hat{A}_i$

$$\hat{A}_i = \frac{R_i - \mu_R}{\sigma_R + \epsilon}, \quad \mu_R = \frac{1}{G}\sum_{j=1}^G R_j, \quad \sigma_R = \sqrt{\frac{1}{G}\sum_{j=1}^G (R_j - \mu_R)^2}$$

这是 PPO 中 Critic 估计 $\hat{A}_i = R_i - V(s_i)$ 的替代物。用组内均值代替 $V(s_i)$。

---

### 5. PPO-Clip 目标

$$\min\left(r_i \hat{A}_i,\ \text{clip}(r_i, 1-\varepsilon, 1+\varepsilon)\hat{A}_i\right)$$

这与 PPO 的 clip 目标完全相同：
- 当 $\hat{A}_i > 0$（好的响应）：clip 防止 $r_i$ 过大（避免策略更新幅度过大）
- 当 $\hat{A}_i < 0$（差的响应）：clip 防止 $r_i$ 过小（避免过度惩罚）
- 综合效果：**保守的策略更新**，不让策略一步走太远

---

### 6. KL 惩罚项

$$-\beta \cdot KL(\pi_\theta \| \pi_{ref})$$

防止策略偏离参考模型（通常是预训练模型或 SFT 模型）过远：
- 与 DPO 中的 $\beta$ 完全对应：$\beta$ 越大，越保守
- 可选：某些 GRPO 变体（如 DAPO）去掉了这一项
- 实现上计算 token-level KL 的均值（不是 sum），避免长度偏差

---

## 从 PPO 到 GRPO 的关键替换

$$\underbrace{\hat{A}_i^{PPO} = R_i - V_\phi(s_i)}_{\text{需要 Critic}} \quad \longrightarrow \quad \underbrace{\hat{A}_i^{GRPO} = \frac{R_i - \mu_R}{\sigma_R}}_{\text{无需 Critic，只需同组样本}}$$

**仅此一处替换，去掉了整个 Critic 网络。其他一切（重要性采样、PPO-Clip、KL 惩罚）保持不变。**

# Cell 5：GRPO vs PPO vs DPO 对比

| 特性 | PPO | DPO | GRPO |
|------|-----|-----|------|
| **Critic 网络** | 需要（同规模 Transformer） | 不需要 | 不需要 |
| **参考模型 $\pi_{ref}$** | 不需要 | 需要（冻结） | 需要（可选，建议保留） |
| **训练模式** | Online（每步采样新数据） | Offline（静态数据集） | Online（每步采样 G 个响应） |
| **数据形式** | 轨迹 + scalar 奖励 | 偏好对 $(y_w, y_l)$ | 采样响应 + scalar 奖励 |
| **优势估计** | $R_i - V_\phi(s_i)$（Critic） | 隐式（由 log prob 比值表达） | $(R_i - \mu_R) / \sigma_R$（组内统计）|
| **策略更新约束** | PPO-Clip | 无显式约束（依赖 $\beta$·KL） | PPO-Clip + KL 惩罚 |
| **适用奖励类型** | 任意 scalar | 仅偏好关系（成对比较） | 任意 scalar |
| **适用场景** | RL 控制任务，游戏 | LLM 偏好对齐（有标注数据）| LLM RLHF（数学/代码，无 Critic） |
| **GPU 内存需求** | Actor + Critic（2x）| Actor + ref_model（2x）| Actor + ref_model（2x）|
| **代表性应用** | OpenAI Five，AlphaGo | Llama-2-Chat，Zephyr | DeepSeek-R1，Qwen2.5 |

---

## 小结

- **PPO**：最通用，但 Critic 对大模型代价太高
- **DPO**：最简洁，但只能处理离线偏好数据
- **GRPO**：PPO 的工程友好版——**同样在线、同样 scalar 奖励，但去掉了 Critic**

GRPO 的代价：需要每个 prompt 采样 G 个响应（计算开销 G 倍），但避免了 Critic 的内存和噪声问题，综合来看对大模型更友好。

# Cell 6：算法伪代码

```
Algorithm GRPO:
  输入：策略 π_θ，参考模型 π_ref，奖励函数 R(·)，
        组大小 G，policy_old 同步频率 K，clip 参数 ε，KL 系数 β
  
  初始化：policy_old = deepcopy(π_θ)
  
  for step = 1, 2, ...:
    
    for each prompt q in mini-batch:
    
      // ===== 采样阶段（无梯度）=====
      1. 从 policy_old 采样 G 个响应（不计算梯度）：
         {o_1, ..., o_G} ~ π_θ_old(· | q)
      
      2. 计算每个响应的奖励：
         R_i = R(o_i)   for i = 1..G
      
      3. 组内归一化优势（无需 Critic！）：
         μ_R = mean({R_i})
         σ_R = std({R_i}) + ε_small
         Â_i = (R_i - μ_R) / σ_R   for i = 1..G
      
      // ===== 损失计算阶段（计算梯度）=====
      4. 计算 per-token average log prob：
         new_lp_i  = mean_t [ log π_θ(o_{i,t} | q, o_{i,<t}) ]      // 需要梯度
         old_lp_i  = mean_t [ log π_θ_old(o_{i,t} | q, o_{i,<t}) ]  // 无梯度
      
      5. 重要性采样比率：
         r_i = exp(new_lp_i - old_lp_i)
      
      6. PPO-Clip Loss（对每个响应）：
         clip_r_i = clamp(r_i, 1-ε, 1+ε)
         L_clip_i = -min(r_i · Â_i, clip_r_i · Â_i)
      
      7. KL 惩罚（token-level mean）：
         KL_i = mean_t [ KL(π_θ(· | o_{i,<t}) || π_ref(· | o_{i,<t})) ]
      
      8. 每个响应的总损失：
         L_i = L_clip_i + β · KL_i
    
    // ===== 更新阶段 =====
    9. 组平均损失：
       L = (1/G) · Σ_i L_i，对 mini-batch 中所有 prompt 求均值
    
    10. 梯度更新：
        θ ← θ - α · ∇_θ L
        （可选：梯度裁剪 clip_grad_norm(θ, max_norm=1.0)）
    
    // ===== 同步阶段 =====
    if step % K == 0:
      policy_old.load_state_dict(π_θ.state_dict())  // 同步 old policy
```

---

## 关键实现细节

1. **采样用 `torch.no_grad()`**：生成响应时不需要梯度，节省内存
2. **old_lp 也用 `torch.no_grad()`**：policy_old 的 log prob 是常数，不参与反传
3. **梯度只来自 new_lp**：通过 `exp(new_lp - old_lp.detach())`，只有 `new_lp` 有梯度
4. **policy_old 同步频率 K**：K 太小（每步同步）等价于 on-policy，浪费计算；K 太大则重要性权重偏差过大，通常 K=1~10
5. **KL 用 token-level mean**：不用 sum，避免长序列 KL 值远大于短序列

# Cell 7：Toy LM 实验说明

## 为什么用 Toy LM 而不是 GPT-2？

| 模型 | 参数量 | CPU 每步 GRPO 耗时 | 300 步总时长 |
|------|--------|--------------------|--------------|
| GPT-2（small） | 117M | ~30 秒 | ~2.5 小时 |
| Toy LM（本节） | ~1M | ~0.5 秒 | ~3 分钟 |

GPT-2 在 CPU 上运行 GRPO 需要数小时，不适合教学演示。Toy LM 牺牲了表达能力，但保留了 Transformer 的完整架构。

---

## Toy 任务设计

**任务**：给定随机 token 序列（5 个 token）作为 prompt，让模型生成包含 **目标 token（ID=42）** 的续写序列（最多 15 个 token）。

**奖励函数**：
$$R(o) = \text{count}(\text{token 42 在生成部分 } o \text{ 中出现的次数})$$

**为什么这样设计**：
- 奖励有梯度信号（partial credit），比 0/1 奖励更易观察训练过程
- 任务简单到可以快速验证 GRPO 机制是否正常工作
- 初始随机模型的期望奖励 = $15 / 1000 = 0.015$（随机猜），训练后应显著高于此

---

## 本节的学习目标

这个 Toy 实验是 **GRPO 机制的单元测试**，不是效果展示。目标：

1. **验证组内归一化优势是否有信号**：训练后奖励应从 ~0.015 提升到 ~0.05-0.1
2. **验证 PPO-Clip 约束有效**：loss 曲线应该平滑收敛，不出现剧烈震荡
3. **验证 KL 惩罚有效**：策略不会完全忘记原始行为（token 42 以外的生成能力）

---

## 注意事项

- Toy LM 只有 2 层 Transformer，无法学习复杂的语言模式
- 训练后奖励提升幅度有限，这是**模型容量限制**，不是 GRPO 的问题
- 真实效果（如 DeepSeek-R1 数学推理准确率大幅提升）见 Cell 15-16 的桥接章节

> 类比：单元测试验证函数逻辑正确，集成测试验证系统性能。Toy LM 是单元测试，DeepSeek-R1 是集成测试。

In [ ]:
class ToyLM(nn.Module):
    """
    2 层 Transformer 解码器，约 1M 参数。
    足够演示 GRPO 的组内归一化机制，同时 CPU 可在几分钟内运行。
    """
    def __init__(self, vocab_size=1000, d_model=128, n_heads=4, n_layers=2, max_len=50):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_len = max_len

        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model*4,
            dropout=0.0, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, input_ids):
        seq_len = input_ids.shape[1]
        positions = torch.arange(seq_len, device=input_ids.device)

        # 因果掩码（只能看到之前的 token）
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        causal_mask = causal_mask.to(input_ids.device)

        x = self.embed(input_ids) + self.pos_embed(positions)
        x = self.transformer(x, mask=causal_mask)
        logits = self.lm_head(x)
        return logits

    def generate(self, prompt_ids, max_new_tokens=15, temperature=1.0):
        """自回归生成（不依赖 HuggingFace）"""
        generated = prompt_ids.clone()

        for _ in range(max_new_tokens):
            if generated.shape[1] >= self.max_len:
                break
            with torch.no_grad():
                logits = self.forward(generated)
            next_logits = logits[:, -1, :] / temperature
            next_token = torch.multinomial(F.softmax(next_logits, dim=-1), 1)
            generated = torch.cat([generated, next_token], dim=1)

        return generated


# 实例化模型
vocab_size = 1000
toy_lm = ToyLM(vocab_size=vocab_size)
ref_model = copy.deepcopy(toy_lm)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

total_params = sum(p.numel() for p in toy_lm.parameters())
print(f"Toy LM 参数量: {total_params/1e6:.2f}M")
print(f"Vocab size: {vocab_size}")

In [ ]:
# 任务：生成包含 token_id=42 ("good" 的替代) 的序列
REWARD_TOKEN_ID = 42

def reward_fn(token_ids):
    """
    奖励 = 目标 token 在生成部分中出现的次数。
    使用 partial credit（每出现一次得 1 分），避免完全稀疏的奖励。

    真实 GRPO 场景（如 DeepSeek-R1）：奖励通常是 0/1（答案正确/错误）
    这里用计数奖励是为了让信号更平滑，便于 Toy 实验观察。
    """
    if isinstance(token_ids, torch.Tensor):
        count = (token_ids == REWARD_TOKEN_ID).sum().item()
    else:
        count = sum(1 for t in token_ids if t == REWARD_TOKEN_ID)
    return float(count)


# 准备一批 prompt（随机 token 序列作为 context）
torch.manual_seed(42)
n_prompts = 20
prompt_len = 5
prompts = [torch.randint(0, vocab_size, (1, prompt_len)) for _ in range(n_prompts)]

# 验证初始奖励分布（随机初始化时约为 0-1 之间）
initial_rewards = []
with torch.no_grad():
    for p in prompts[:5]:
        gen = toy_lm.generate(p, max_new_tokens=15)
        r = reward_fn(gen[0, prompt_len:])  # 只计算生成部分
        initial_rewards.append(r)

print(f"随机初始化基准奖励: {np.mean(initial_rewards):.4f} +/- {np.std(initial_rewards):.4f}")
print(f"理论期望（vocab={vocab_size}，len=15）: {15/vocab_size:.4f}")
print(f"目标奖励（训练后）: ~0.05+")

In [ ]:
def compute_log_probs(model, token_ids):
    """
    计算序列的 per-token average log prob。

    用 mean(-1) 而非 sum(-1)：避免序列长度偏差。
    注意：此处 Toy LM 无 prompt/response 区分，计算整个序列的 log prob。
    在真实 LLM 场景中，应只计算 response 部分（用 attention mask 或切片）。
    """
    logits = model(token_ids)  # [batch, seq, vocab]
    log_probs = F.log_softmax(logits, dim=-1)
    # 每个位置的 log prob = logits[t] 对应 token[t+1] 的 log prob
    token_lp = log_probs[:, :-1].gather(-1, token_ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    return token_lp.mean(-1)  # [batch]，mean over tokens（非 sum）


def compute_group_advantages(rewards):
    """
    组内归一化优势。

    GRPO 的核心：用组内均值替代 Critic 的基准线。
    - mean(rewards)：组内基准（相当于 Critic 的估计）
    - std(rewards)：归一化，使得 advantages 的量级稳定
    """
    rewards_t = torch.tensor(rewards, dtype=torch.float32)
    mean = rewards_t.mean()
    std  = rewards_t.std() + 1e-8
    advantages = (rewards_t - mean) / std
    return advantages


def compute_kl_from_logprobs(policy, ref_model, token_ids):
    """
    计算 KL(pi_theta || pi_ref) 的 token-level mean。

    为什么用 mean 不用 sum：不同响应长度不同，sum 引入长度偏差。

    F.kl_div 参数说明：
    - input  = log(Q)（参考模型的 log prob）
    - target = log(P)（策略模型的 log prob），配合 log_target=True
    - 这里计算 KL(pi_theta || pi_ref)，即 P=pi_theta，Q=pi_ref
    注意：F.kl_div(log_q, log_p, log_target=True) 计算 KL(P||Q)
    """
    with torch.no_grad():
        ref_logits   = ref_model(token_ids)
        ref_log_prob = F.log_softmax(ref_logits, dim=-1)

    policy_logits   = policy(token_ids)
    policy_log_prob = F.log_softmax(policy_logits, dim=-1)

    # KL(pi_theta || pi_ref) = E[pi_theta * log(pi_theta/pi_ref)]
    # F.kl_div(input=log_q, target=log_p, log_target=True) = KL(P||Q)
    kl_per_token = F.kl_div(
        ref_log_prob,           # log Q = log pi_ref
        policy_log_prob,        # log P = log pi_theta
        reduction='none',
        log_target=True
    ).sum(-1)  # sum over vocab dim

    return kl_per_token.mean()  # mean over sequence and batch


# 快速验证工具函数
test_input = torch.randint(0, vocab_size, (1, 10))
test_lm = ToyLM(vocab_size=vocab_size)
lp = compute_log_probs(test_lm, test_input)
print(f"compute_log_probs 输出 shape: {lp.shape}，值: {lp.item():.4f}")

test_rewards = [0.0, 0.0, 1.0, 2.0]
advs = compute_group_advantages(test_rewards)
print(f"compute_group_advantages([0,0,1,2]): {advs.tolist()}")
print("（均值应为 0，sum 应接近 0）")

In [ ]:
# ============================================================
# 关键：policy_old 的初始化与同步机制
# ============================================================
# GRPO 需要一个"旧策略"来计算重要性采样比率 r = pi_theta / pi_theta_old
# 与 PPO 相同，每 K 步同步一次 policy_old

policy = ToyLM(vocab_size=vocab_size)
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)

# 初始化 policy_old（深拷贝，独立于 policy）
policy_old = copy.deepcopy(policy)
policy_old.eval()  # policy_old 只用于推理

K = 10  # 每 K 步同步一次 policy_old

# 说明：在训练循环中，每 K 步执行：
# policy_old.load_state_dict(policy.state_dict())
# 这保证了 policy_old 不会与 policy 偏差过大

print("policy_old 初始化完成")
print(f"同步频率 K = {K}（每 {K} 步更新一次 policy_old）")
print()
print("内存结构说明：")
print("  policy      - 训练中的策略，参数会更新")
print("  policy_old  - 采样用，每 K 步从 policy 同步")
print("  ref_model   - 冻结的参考模型，用于 KL 惩罚，永不更新")

In [ ]:
def grpo_update(policy, policy_old, ref_model, optimizer, prompts_batch,
                G=8, beta=0.01, eps=0.2):
    """
    GRPO 核心更新函数。

    参数说明：
    - policy:      被训练的策略模型（会更新参数）
    - policy_old:  旧策略（外部每 K 步同步一次，不在此函数中更新）
    - ref_model:   冻结的参考模型（用于 KL 惩罚）
    - G:           组大小（每个 prompt 采样 G 个响应）
    - beta:        KL 惩罚系数
    - eps:         PPO clip 参数

    返回：(loss_value, mean_reward_of_last_prompt)
    """
    policy.train()
    n_prompts_in_batch = len(prompts_batch)

    # 累计损失（用列表收集，最后合并，避免原地操作影响梯度图）
    all_losses = []
    last_rewards = [0.0]  # 用于监控，记录最后一个 prompt 的奖励

    for prompt in prompts_batch:
        # Step 1: 从 policy_old 采样 G 个响应（无梯度）
        responses = []
        for _ in range(G):
            with torch.no_grad():
                gen = policy_old.generate(prompt, max_new_tokens=15, temperature=1.0)
                responses.append(gen)

        # Step 2: 计算每个响应的奖励
        rewards = [reward_fn(r[0, prompt.shape[1]:]) for r in responses]
        last_rewards = rewards  # 记录用于监控

        # Step 3: 组内归一化优势（GRPO 核心）
        advantages = compute_group_advantages(rewards)

        # Step 4-6: 对每个响应计算 PPO-Clip + KL 惩罚
        group_losses = []
        for i, (resp, adv) in enumerate(zip(responses, advantages)):
            if resp.shape[1] < 2:
                continue

            # 重要性采样比率（per-token average log prob 之差的 exp）
            new_lp = compute_log_probs(policy, resp)          # 有梯度
            with torch.no_grad():
                old_lp = compute_log_probs(policy_old, resp)  # 无梯度

            ratio = (new_lp - old_lp).exp()  # r_i

            # PPO-Clip Loss
            clip_ratio = ratio.clamp(1 - eps, 1 + eps)
            adv_val = adv.item()  # scalar advantage for this response
            policy_loss = -torch.min(ratio * adv_val, clip_ratio * adv_val).mean()

            # KL 惩罚（token-level mean）
            kl = compute_kl_from_logprobs(policy, ref_model, resp)

            group_losses.append(policy_loss + beta * kl)

        if group_losses:
            all_losses.append(torch.stack(group_losses).mean())

    if not all_losses:
        return 0.0, 0.0

    # 梯度更新
    total_loss = torch.stack(all_losses).mean()
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    optimizer.step()

    return total_loss.item(), float(np.mean(last_rewards))


# 快速单步测试
test_policy = ToyLM(vocab_size=vocab_size)
test_ref = copy.deepcopy(test_policy)
test_ref.eval()
for p in test_ref.parameters():
    p.requires_grad = False
test_old = copy.deepcopy(test_policy)
test_old.eval()
test_opt = torch.optim.Adam(test_policy.parameters(), lr=1e-3)

loss_val, r_val = grpo_update(test_policy, test_old, test_ref, test_opt,
                               [prompts[0], prompts[1]], G=4)
print(f"单步测试通过！Loss={loss_val:.4f}, Mean Reward={r_val:.4f}")

In [ ]:
# 重新初始化（保证可重现）
torch.manual_seed(42)
policy = ToyLM(vocab_size=vocab_size)
ref_model = copy.deepcopy(policy)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

policy_old = copy.deepcopy(policy)
policy_old.eval()

optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)

K = 10          # policy_old 同步频率
n_steps = 300
batch_size = 5  # 每步使用 5 个 prompt
G = 8           # 每个 prompt 采样 G=8 个响应

reward_history = []
loss_history = []

print(f"GRPO 训练开始（{n_steps} 步，G={G}，K={K}）")
print(f"每步采样 G x batch_size = {G * batch_size} 个响应序列")
print("-" * 55)

for step in range(n_steps):
    # 随机选取一批 prompt
    batch_idx = np.random.choice(len(prompts), batch_size, replace=False)
    batch = [prompts[i] for i in batch_idx]

    loss, mean_reward = grpo_update(
        policy, policy_old, ref_model, optimizer, batch, G=G
    )

    reward_history.append(mean_reward)
    loss_history.append(loss)

    # 每 K 步同步 policy_old
    if (step + 1) % K == 0:
        policy_old.load_state_dict(policy.state_dict())

    if (step + 1) % 50 == 0:
        recent_reward = np.mean(reward_history[-20:])
        print(f"Step {step+1:4d} | Loss: {loss:.4f} | Avg Reward (last 20): {recent_reward:.4f}")

print("-" * 55)
print(f"\n训练完成！")
print(f"初始奖励（前 20 步均值）: {np.mean(reward_history[:20]):.4f}")
print(f"最终奖励（后 20 步均值）: {np.mean(reward_history[-20:]):.4f}")
print(f"随机基准（期望）:         {15/vocab_size:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- 左图：奖励曲线 ---
window = 20
axes[0].plot(reward_history, alpha=0.3, color='green', label='每步奖励')
if len(reward_history) >= window:
    smooth = np.convolve(reward_history, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(reward_history)), smooth,
                 color='green', lw=2, label=f'{window} 步滑动均值')
axes[0].axhline(y=15/vocab_size, color='gray', linestyle='--', alpha=0.7,
                label=f'随机基准 ({15/vocab_size:.4f})')
axes[0].set_xlabel('训练步数')
axes[0].set_ylabel('平均奖励（目标 token 出现次数）')
axes[0].set_title('GRPO 训练曲线（Toy LM）')
axes[0].legend()
axes[0].grid(alpha=0.3)

# --- 右图：奖励分布对比 ---
torch.manual_seed(0)
init_model = ToyLM(vocab_size=vocab_size)  # 新随机初始化

initial_dist = []
final_dist = []

with torch.no_grad():
    for p in prompts:
        gen_init = init_model.generate(p, max_new_tokens=15)
        r_init = reward_fn(gen_init[0, prompt_len:])
        initial_dist.append(r_init)

        gen_final = policy.generate(p, max_new_tokens=15)
        r_final = reward_fn(gen_final[0, prompt_len:])
        final_dist.append(r_final)

bins = range(0, max(max(initial_dist, default=1), max(final_dist, default=1)) + 3)
axes[1].hist(initial_dist, bins=bins, alpha=0.6, color='steelblue',
             label=f'训练前 (mean={np.mean(initial_dist):.3f})', density=True)
axes[1].hist(final_dist,   bins=bins, alpha=0.6, color='green',
             label=f'训练后 (mean={np.mean(final_dist):.3f})',   density=True)
axes[1].set_xlabel('目标 token 出现次数')
axes[1].set_ylabel('频率（归一化）')
axes[1].set_title('奖励分布：训练前 vs 训练后')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('GRPO on Toy LM（组相对优势 + PPO-Clip，无 Critic）', fontsize=12)
plt.tight_layout()
plt.savefig('grpo_toy_lm.png', dpi=100, bbox_inches='tight')
plt.show()
print("图像已保存至 grpo_toy_lm.png")

# Cell 15：桥接章节（一）—— 从 Toy LM 到真实 LLM

## 本节说明

本 notebook 使用 Toy LM（~1M 参数，vocab=1000）演示了 GRPO 的核心机制：
1. 对同一 prompt 采样 G 个响应
2. 用组内均值归一化奖励，替代 Critic 网络
3. PPO-Clip 损失 + KL 惩罚稳定训练

现在来看：如果把 Toy LM 替换为真实 LLM（如 DeepSeek-R1-7B），**哪些部分不变，哪些需要改变**？

---

## 组件映射表

| Toy LM 组件 | 真实 LLM（DeepSeek-R1）对应物 | 是否变化 |
|---|---|---|
| `ToyLM`（2 层，1M 参数）| 7B/70B Transformer | 架构相同，规模不同 |
| `vocab_size=1000` | 真实 tokenizer（~32K-128K vocab）| 规模不同，机制相同 |
| `reward_fn`（token 计数）| 数学推理评判器、代码测试器、安全过滤器 | **完全替换** |
| `G=8` 组采样 | G=16 或更大（需要更多 GPU）| 超参调整 |
| `compute_group_advantages` | **完全不变** | ✅ 核心机制 |
| `compute_log_probs`（mean）| **完全不变** | ✅ |
| `grpo_update`（PPO-Clip + KL）| **完全不变** | ✅ 核心机制 |
| `policy_old` 同步（K=10）| **完全不变** | ✅ |
| 序列长度（15 tokens）| 512-2048 tokens（思维链推理）| 长度增大 |
| `torch.no_grad()` 推理 | vLLM 等推理框架加速 | 实现加速 |

---

## 关键洞察

> **GRPO 的核心（组内归一化 + PPO-Clip + KL 惩罚）与模型规模无关。Toy LM 实验验证了这个机制的正确性。**

从 Toy LM 迁移到 7B LLM，主要工程挑战在于：

1. **分布式训练**：7B 模型需要多 GPU，需要用 DeepSpeed/FSDP 切分参数
2. **推理加速**：每步采样 G=16 条 2048-token 响应，需要 vLLM 等高效推理引擎
3. **奖励函数工程**：数学题验证器（调用符号计算引擎如 SymPy）、代码执行沙箱
4. **超参数调优**：$\beta$（KL 系数）、$G$（组大小）、学习率等需要针对大模型重新调整

---

## 真实场景中的 `compute_log_probs`

在 Toy LM 中，我们用整个序列（prompt + response）计算 log prob，这是简化。

在真实 LLM 中，应只计算 **response 部分** 的 log prob（prompt 的 log prob 是常数，不影响梯度方向，但会影响数值稳定性）：

```python
# 真实场景的 compute_log_probs（伪代码）
def compute_log_probs_real(model, full_ids, prompt_len):
    logits = model(full_ids)  # [batch, seq, vocab]
    log_probs = F.log_softmax(logits, dim=-1)
    # 只取 response 部分的 log prob
    response_ids = full_ids[:, prompt_len:]          # response tokens
    response_logits = log_probs[:, prompt_len-1:-1]  # 对应的 logits
    token_lp = response_logits.gather(-1, response_ids.unsqueeze(-1)).squeeze(-1)
    return token_lp.mean(-1)  # per-token mean，只在 response 上
```

---

## 开源实现参考

如果你想在真实 LLM 上运行 GRPO，以下开源框架已经做好了工程化：

- **TRL（HuggingFace）**：`GRPOTrainer` 类，与 HuggingFace 生态无缝集成
- **OpenRLHF**：支持 70B+ 模型的分布式 GRPO 训练
- **veRL**：字节跳动开源的大规模 RLHF 框架，支持 GRPO

这些框架实现的 `GRPOTrainer` 核心逻辑与我们的 `grpo_update` 函数在数学上完全一致。

# Cell 16：桥接章节（二）—— DeepSeek-R1 中的 GRPO

## DeepSeek-R1：GRPO 的标志性应用

2025 年 1 月，DeepSeek 发布了 DeepSeek-R1 技术报告，将 GRPO 推上了 LLM 训练的主流视野。

### 任务设置

- **任务**：数学推理（AIME、MATH 等竞赛题）+ 代码生成
- **奖励函数**：
  - 答案正确：$R = 1$（通过验证器或代码沙箱验证）
  - 答案错误：$R = 0$
  - 格式奖励：响应包含 `<think>...</think>` 思维链格式额外加分
- **组大小**：$G = 16$（每道题生成 16 个解答）
- **序列长度**：最长 32768 tokens（长思维链推理）

### 关键发现

训练过程中出现了 **涌现行为（emergent behavior）**：

> 在没有任何人工干预的情况下，模型自发学会了「反思」——在推理中途发现错误后，自动回退并尝试新的解题路径，类似于人类思考时的 "wait, let me reconsider"。

这被视为 GRPO 推动 LLM 获得「推理能力」的直接证据。

---

## 为什么 GRPO 比 PPO 更适合大模型？

| 问题 | PPO 的解法 | GRPO 的解法 |
|------|-----------|------------|
| Critic 内存 | 需要与 Actor 同等规模的 Critic（内存 2x）| 无 Critic，节省 ~50% 内存 |
| Critic 噪声 | token-level value 估计噪声大 | 组内统计，简单但稳定 |
| 训练稳定性 | PPO-Clip | PPO-Clip + KL 惩罚（双重保护）|

在 70B 参数的模型上：
- PPO：70B Actor + 70B Critic + 70B ref_model = 210B 参数同时常驻 GPU（完全不可行）
- GRPO：70B policy + 70B ref_model = 140B 参数（仍然挑战大，但可行）

---

## GRPO 的局限性

### 1. 组采样计算代价

每步 GRPO 需要对每个 prompt 生成 G 个完整响应。G=16 意味着每步的推理计算量是普通前向传播的 16 倍。

缓解方法：
- 使用 vLLM 等批量推理引擎，通过连续批处理（continuous batching）减少空闲时间
- 异步采样：采样和训练并行进行

### 2. 奖励 Hacking 风险

模型可能发现奖励函数的漏洞：
- 数学题：输出大量符合格式但内容重复的「推理步骤」来占满上下文，凑出正确答案
- 代码：生成通过测试用例但不通用的 hardcode 解法

KL 惩罚项 $\beta \cdot KL(\pi_\theta || \pi_{ref})$ 是对抗 hacking 的防线之一，但不能完全消除。

### 3. 奖励函数设计难度

GRPO 的训练质量完全取决于奖励函数的质量：
- 0/1 稀疏奖励（答案对/错）：信号稀少，但干净，避免 hacking
- 连续奖励（如本 notebook 的 token 计数）：信号丰富，但易被 hacking
- 过程奖励（每个推理步骤单独评分）：最细粒度，但最难设计

DeepSeek-R1 选择了 0/1 + 格式奖励的组合，在干净信号和训练效率之间取得了平衡。

---

## GRPO 的变体与未来方向

| 变体 | 与 GRPO 的区别 | 动机 |
|------|---------------|------|
| **DAPO**（ByteDance, 2025）| 去掉 clip，使用更激进的更新 | 认为 clip 过于保守，限制了探索 |
| **Dr. GRPO** | 修正 GRPO 中的长度偏差（std 计算问题）| 短序列和长序列的 std 不可比 |
| **REINFORCE++** | GRPO + 更精细的方差减小技巧 | 进一步提升采样效率 |
| **Process GRPO** | 对每个推理步骤单独计算组内优势 | 更细粒度的过程奖励 |

---

## 总结：GRPO 的优雅之处

```
GRPO = PPO 的稳定更新机制
     + 组内统计作为 baseline（替代 Critic）
     + KL 惩罚（防止偏离参考模型）
     - Critic 网络（大模型无法承受的内存开销）
```

GRPO 的核心洞察极其简单：**不需要知道「绝对好」，只需要知道「相对于同组，是好还是坏」**。

这个简单的洞察，配合 PPO 成熟的训练稳定性技巧，使得在消费级 GPU 集群上训练推理能力强大的 LLM 成为可能。

---

## 系列总结

| Notebook | 算法 | 核心思想 |
|----------|------|----------|
| 01_TRPO | TRPO | 信任域约束，二阶优化，保守更新 |
| 02_PPO | PPO | Clip 替代信任域，一阶优化，实用高效 |
| 03_DPO | DPO | 偏好数据直接对齐，无需 RL，极简实现 |
| 04_GRPO | GRPO | 组内基准替代 Critic，无 Critic 的在线 RL |

四个算法构成了现代 LLM 对齐技术的主要技术路线：从强化学习出发（TRPO→PPO），再到偏好学习（DPO），最后到无 Critic 的在线学习（GRPO）。不同场景、不同约束下，选择合适的算法是工程实践中的关键判断。